# 24 · Jev outputs as features for a classic ML model

## The problem

You already have a churn model. It runs on structured columns from your database: **tenure, monthly spend, support
tickets**. It's fast, explainable, and your data team trusts it. But the customer's **last message** often says
something those columns can't: *"we're evaluating alternatives"*, *"my boss is asking why we pay for this"*.
Your logistic regression can't read.

You don't want to replace the model with an LLM. You want to give it **a couple of new numeric columns** that capture
what the text says.

## Why Jev

Jev turns text into **typed, calibrated numbers**, which is exactly what a feature column needs: `Noul` gives a
probability (*does the customer signal they'll leave?*), `Score` gives a position on a rubric (*how frustrated?*).
Both come back in one call, in about 100 ms, for a fraction of a cent per row, so you can backfill a whole table and
score new rows in real time. The existing model stays in charge; Jev just adds columns.

## What we'll build

```
customer row ──────────────────────────────────────────────┐
  tenure, spend, tickets                                    ├──► logistic regression ──► P(churn)
  last_message ──► Jev: Noul "intends to leave?"  ─► 0..1 ──┤
                        Score "frustration" 0..2  ─► 0..1 ──┘

evaluate:  same model, same held-out splits,  baseline columns  vs  baseline + Jev columns
```

## What you'll learn
- How to use `Noul` and `Score` answers as **feature columns** for an ordinary ML model.
- The evaluation habits that keep you honest: held-out splits, a baseline, train-only scaling, features that never see the label.
- Why a big jump on a **24-row toy** is a demonstration of the *method*, not a result, and what you'd need to trust it on real data.

> **This is a teaching toy, not a benchmark.** The 24 customers are hand-written to illustrate the idea. Step 5 explains
> exactly why the accuracy jump you'll see is inflated.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #18.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell loads `TYPESAFE_API_KEY` from the `.env` file at the repo root, creates the Jev client `jev`, and imports
NumPy and pandas for the table and the model. We use **`Noul`** (with `NoulCriteria`) and **`Score`**.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice
import numpy as np
import pandas as pd

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · A toy customer table

24 customers: three structured columns, the free-text `last_message`, and the label `churned` (1 = left).

Read a few of the messages before moving on. We wrote them by hand to make the idea easy to see, and that has a
consequence: **almost every churner's message openly says they're leaving** (*"Please cancel..."*, *"Not renewing..."*),
and almost no one else's does. Keep that in mind; it comes back in Step 5.

In [2]:
ROWS = [
    # tenure_months, monthly_spend, tickets_30d, last_message, churned
    (24, 120, 0, "Loving the new reporting feature, the team uses it daily.", 0),
    (3, 40, 1, "How do I invite my colleague?", 0),
    (36, 300, 2, "Can you send last quarter's invoices to our finance team?", 0),
    (12, 80, 4, "Third outage this month. We're evaluating alternatives.", 1),
    (6, 45, 0, "Please cancel at the end of the billing period.", 1),
    (18, 150, 1, "Is there an API endpoint for exporting projects?", 0),
    (2, 30, 3, "This is way too complicated, nothing works the way I expect.", 1),
    (40, 500, 1, "We'd like to add 20 more seats next month.", 0),
    (9, 60, 0, "Found a cheaper tool that does the same thing. How do I export my data?", 1),
    (15, 90, 2, "The sync bug is annoying but support has been great, thanks.", 0),
    (5, 35, 0, "Our startup is shutting down, so we won't need this anymore.", 1),
    (28, 220, 3, "Performance has been slow, any update on the fix?", 0),
    (7, 50, 1, "Honestly disappointed. Not renewing unless pricing changes.", 1),
    (20, 110, 0, "Great product. Any plans for a mobile app?", 0),
    (11, 75, 5, "Still no fix after two weeks. My boss is asking why we pay for this.", 1),
    (30, 260, 0, "Can we get a demo of the new admin controls for our IT team?", 0),
    (4, 40, 2, "Trial ends soon and I still can't get the integration working, about to give up.", 1),
    (22, 140, 1, "Small typo on the billing page, just FYI.", 0),
    (14, 95, 0, "We've migrated most projects elsewhere; please downgrade us to free.", 1),
    (33, 280, 2, "Two bugs reported, both fixed quickly. Appreciate it.", 0),
    (8, 55, 1, "Is there a discount for annual billing?", 0),
    (16, 100, 3, "Your latest update broke our workflow and we are reconsidering.", 1),
    (26, 180, 0, "Could you enable SSO for our workspace?", 0),
    (10, 70, 0, "Thinking of switching, the competitor has the feature we need.", 1),
]
df = pd.DataFrame(ROWS, columns=["tenure_months", "monthly_spend", "tickets_30d", "last_message", "churned"])
df.head()

,tenure_months,monthly_spend,tickets_30d,last_message,churned
0,24,120,0,"Loving the new reporting feature, the team use...",0
1,3,40,1,How do I invite my colleague?,0
2,36,300,2,Can you send last quarter's invoices to our fi...,0
3,12,80,4,Third outage this month. We're evaluating alte...,1
4,6,45,0,Please cancel at the end of the billing period.,1


## Step 2 · Jev turns text into features

Two questions, both about the **message text only**:

- `intends_to_leave` is a **`Noul`**: `.noul` is P(yes), from 0 to 1. `NoulCriteria(true=..., false=...)` pins down
  exactly what counts as *yes* and what counts as *no*. Here that matters: a complaint is **not** the same as an intent
  to leave, and we say so explicitly.
- `frustration` is a **`Score`** over a 3-level rubric (level 0..2). `.score` is a continuous position on it; we
  divide by 2 so the feature runs from 0 to 1.

Both go in **one** `system_one` call per row. Jev never sees the `churned` column, so the label can't leak into the
features *through Jev*. (Whether it leaks through the *text itself* is a different question, see Step 5.)

> **Workshop note:** the original `Noul` had no `criteria`, only the question. We added `NoulCriteria` so "yes" and
> "no" are spelled out. It's a small wording change, but it's what makes a feature *stable*: you want the column to
> mean the same thing on every row and every day.

This cell only **defines** the questions and the feature function.

In [3]:
FEATURE_QUESTIONS = {
    "intends_to_leave": Noul(
        instructions="Does the customer signal they intend to cancel, downgrade or switch providers?",
        criteria=NoulCriteria(
            true="The customer says or clearly implies they will cancel, downgrade, stop paying, or move to another provider",
            false="No sign of leaving: questions, praise, requests, or complaints without any hint of cancelling or switching",
        ),
    ),
    "frustration": Score(instructions="How frustrated is the customer?",
                         criteria=["Positive or neutral", "Mildly annoyed", "Clearly frustrated"]),
}


def jev_features(text, client):
    """Return (P(intends to leave), frustration on 0..1) for one message."""
    if "FEATURE_QUESTIONS" not in globals():
        raise RuntimeError("FEATURE_QUESTIONS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")
    r = client.system_one(state=text, questions=FEATURE_QUESTIONS)
    return r.nouls["intends_to_leave"].noul, r.scores["frustration"].score / 2

### Backfill the two new columns (24 Jev calls)

In [4]:
df[["jev_leave", "jev_frustration"]] = [jev_features(t, jev) for t in df["last_message"]]
pd.set_option("display.max_colwidth", 70)
df[["last_message", "jev_leave", "jev_frustration", "churned"]].round(2)

,last_message,jev_leave,jev_frustration,churned
0,"Loving the new reporting feature, the team uses it daily.",0.02,0.00,0
1,How do I invite my colleague?,0.01,0.00,0
2,Can you send last quarter's invoices to our finance team?,0.01,0.00,0
3,Third outage this month. We're evaluating alternatives.,0.79,1.00,1
4,Please cancel at the end of the billing period.,0.99,0.01,1
5,Is there an API endpoint for exporting projects?,0.02,0.00,0
6,"This is way too complicated, nothing works the way I expect.",0.11,1.00,1
7,We'd like to add 20 more seats next month.,0.01,0.00,0
8,Found a cheaper tool that does the same thing. How do I export my ...,0.91,0.08,1
9,"The sync bug is annoying but support has been great, thanks.",0.03,0.46,0


**What to notice**
- `jev_leave` is high for messages like *"Please cancel..."* and *"Thinking of switching..."*, and near 0 for questions and praise.
- Complaints without a threat to leave (*"Performance has been slow, any update on the fix?"*) get a clearly
  raised **frustration** score but a **low** leave probability. That's the `NoulCriteria` doing its job: two different signals.
- Look down the `churned` column next to `jev_leave`: they line up for most rows. The churners Jev scores low on
  leaving (*"nothing works the way I expect"*, *"my boss is asking why we pay"*) get maximum frustration instead, so
  between the two columns almost every churner stands out. On this toy, that's because the messages were written to
  make the label obvious. Remember this for Step 5.

## Step 3 · A tiny logistic regression in NumPy

A plain logistic regression, written out in a few lines so nothing is hidden:

- `fit_logreg()` runs gradient descent with a little L2 regularisation (it stops weights from blowing up on tiny data).
- `evaluate()` **standardises with the training rows' mean and std only**, fits, then scores the held-out rows. Using
  test-set statistics for scaling is a classic subtle leak.
- It returns **accuracy** (share of correct yes/no calls at 0.5) and **log loss** (how good the probabilities are;
  lower is better, and it punishes confident mistakes hard).

This cell only **defines** the functions.

In [5]:
def fit_logreg(X, y, lr=0.1, steps=4000, l2=0.01):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(X @ w + b)))
        w -= lr * (X.T @ (p - y) / len(y) + l2 * w)
        b -= lr * np.mean(p - y)
    return w, b


def evaluate(cols, train, test):
    mu, sd = train[cols].mean(), train[cols].std() + 1e-9          # scale using TRAIN stats only
    Xtr, Xte = ((train[cols] - mu) / sd).values, ((test[cols] - mu) / sd).values
    w, b = fit_logreg(Xtr, train["churned"].values)
    p = 1 / (1 + np.exp(-(Xte @ w + b)))
    y = test["churned"].values
    acc = np.mean((p >= 0.5) == y)
    logloss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
    return acc, logloss

## Step 4 · Held-out evaluation

Same model, same splits, two feature sets:

| Feature set | Columns |
|---|---|
| `BASE` | tenure, spend, tickets |
| `WITH_JEV` | the same three **plus** `jev_leave` and `jev_frustration` |

Each split trains on 16 customers and tests on the 8 it never saw. We repeat over 5 random shuffles so a single lucky
split doesn't decide the story.

In [6]:
BASE = ["tenure_months", "monthly_spend", "tickets_30d"]
WITH_JEV = BASE + ["jev_leave", "jev_frustration"]
SEEDS = range(5)


def compare(feature_sets):
    """Mean held-out accuracy and log loss for each named feature set, over the same SEEDS splits."""
    rows = []
    for seed in SEEDS:
        shuffled = df.sample(frac=1, random_state=seed)
        train, test = shuffled.iloc[:16], shuffled.iloc[16:]
        row = {"seed": seed}
        for name, cols in feature_sets.items():
            row[f"{name}_acc"], row[f"{name}_logloss"] = evaluate(cols, train, test)
        rows.append(row)
    return pd.DataFrame(rows).set_index("seed")


res = compare({"base": BASE, "jev": WITH_JEV})
print(res.round(2))
print("\nmean:", res.mean().round(2).to_dict())

      base_acc  base_logloss  jev_acc  jev_logloss
seed                                              
0         0.88          0.30     1.00         0.03
1         0.62          0.43     1.00         0.08
2         0.50          0.57     1.00         0.06
3         0.62          0.46     1.00         0.09
4         0.62          1.33     0.88         0.25

mean: {'base_acc': 0.65, 'base_logloss': 0.62, 'jev_acc': 0.98, 'jev_logloss': 0.1}


**What to notice**
- With Jev's two columns, mean held-out accuracy jumps from roughly two-thirds to nearly perfect, and log loss falls
  sharply.
- The baseline swings a lot from split to split. With only 8 test rows, **one customer is 12.5 points of accuracy**.
- Before celebrating, read the next step.

## Step 5 · Why that jump is inflated (read this before quoting any number)

A jump from roughly two-thirds accuracy to nearly perfect would be extraordinary on real churn data. Here it mostly reflects **how the toy
was built**, not how good the features are. Four reasons:

1. **The label is written in the text.** We wrote the messages by hand, and the churners say *"cancel"*,
   *"not renewing"*, *"switching"*. A feature that detects "this message says they're leaving" is then almost a copy of
   the label. Real customers rarely announce it; most churn silently.
2. **It's partly a timing leak.** *"Please cancel at the end of the billing period"* isn't a *predictor* of churn, it
   *is* the churn event. In production you must only use messages sent **before** the moment you make the prediction,
   or you're predicting the past.
3. **Tiny test sets.** 8 rows per split means each mistake moves accuracy by 12.5 points, and the 5 splits reuse the same
   24 rows, so they're not 5 independent experiments.
4. **A weak baseline by construction.** The structured columns in the toy were not designed to be very predictive, so
   almost anything that reads the text will look like a huge gain.

**A quick test of reason 1.** If the label really is "written in the message", then a dumb keyword rule should do
nearly as well as Jev. The next cell adds a single 0/1 column, *"does the message contain a leaving word?"*, and runs
the exact same evaluation.

In [7]:
LEAVE_WORDS = r"cancel|not renewing|switch|alternatives|elsewhere|cheaper|downgrade|shutting down|give up|reconsider"
df["kw_leave"] = df["last_message"].str.lower().str.contains(LEAVE_WORDS).astype(float)
print("keyword rule flags", int(df["kw_leave"].sum()), "of", len(df), "messages;",
      "agrees with the label on", int((df["kw_leave"] == df["churned"]).sum()), "of", len(df))

res3 = compare({"base": BASE, "keyword": BASE + ["kw_leave"], "jev": WITH_JEV})
print("\nmean:", res3.mean().round(2).to_dict())

keyword rule flags 9 of 24 messages; agrees with the label on 22 of 24



mean: {'base_acc': 0.65, 'base_logloss': 0.62, 'keyword_acc': 0.95, 'keyword_logloss': 0.15, 'jev_acc': 0.98, 'jev_logloss': 0.1}


**What to notice**
- A ten-word regex, written by someone who **read the toy messages**, also lifts the baseline a lot. That's the proof
  that the toy's text basically contains the label.
- So the honest takeaway from this notebook is **the method**, not the number: add Jev columns, keep the same model
  and splits, scale with training stats only, compare against a baseline *and* a cheap text baseline.
- Where Jev should really earn its place is on messages a keyword list can't handle: hedged wording, sarcasm, other
  languages. Even Jev is only lukewarm on *"my boss is asking why we pay for this"*: subtle signals are hard, and only
  **real data** tells you how much they're worth.

**What a real evaluation needs:** thousands of rows; messages strictly from *before* the prediction date; a
time-based split (train on older customers, test on newer ones); a text baseline (keywords or TF-IDF); calibration
checks on the predicted probabilities; and the per-row Jev cost weighed against the lift.

## Step 6 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `jev_features()` and the
logistic regression, and an HTML card with the churn risk and the two Jev features. The feature and evaluation logic
above is untouched.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`customer_html()` takes one customer, gets the two Jev features for the message, fits the model on **all 24 rows**
(fine for a demo, since there's nothing held out to protect here), and shows the predicted churn risk. Our example is
a customer who is happy but under budget pressure: a message with no complaint and no explicit cancel.

In [9]:
def customer_html(tenure, spend, tickets, message):
    leave, frus = jev_features(message, jev)
    full = df.copy()
    w, b = fit_logreg(((full[WITH_JEV] - full[WITH_JEV].mean()) / (full[WITH_JEV].std() + 1e-9)).values,
                      full["churned"].values)
    x = (pd.Series([float(tenure), float(spend), float(tickets), leave, frus], index=WITH_JEV)
         - full[WITH_JEV].mean()) / (full[WITH_JEV].std() + 1e-9)
    p = 1 / (1 + np.exp(-(x.values @ w + b)))
    col = RED if p > 0.6 else AMBER if p > 0.35 else SAGE
    return box(f'<div style="{CARD}border-left:5px solid {col};">{badge(f"churn risk {p:.0%}", col)}'
               f'<div style="margin:6px 0;">"{message}"</div>'
               f'{meter("jev: intends to leave", leave)}{meter("jev: frustration", frus)}</div>')


display(HTML(customer_html(20, 150, 1, "We love it, but the CFO wants us to cut costs, so we're reviewing all tools.")))

## Step 7 · Try it yourself

Change the numbers and the message and press **Predict** (needs a live kernel). Good ones to try:
- The pre-filled ticket complaint (*"Support keeps closing my ticket..."*) → high frustration; watch how much
  churn risk that adds even without an explicit threat to leave.
- Tenure `36`, spend `300`, tickets `0`, *"Please cancel our account at the end of this month."* → `jev_leave` near 1, yet
  the churn risk is only moderate: a 36-month, $300 customer is beyond almost every churner in the toy, so the
  structured columns pull the other way. A good moment to talk about how little 24 rows can teach a model.
- Tenure `2`, spend `30`, tickets `5`, *"Thanks, the new export feature is exactly what we needed!"* → leave probability near
  0 and a low churn risk despite the many tickets.
- *"My boss is asking why we pay for this."* → no "leaving word", so the keyword rule
  would give 0. Jev gives a low-to-modest leave probability plus noticeable frustration.

In [10]:
form_app([("tenure", "Tenure (months)", "12", 1), ("spend", "Monthly spend ($)", "90", 1),
          ("tickets", "Tickets in last 30 days", "2", 1),
          ("message", "Last message", "Support keeps closing my ticket without fixing anything.", 2)],
         customer_html, "Predict")

Output()

## Recap and exercises

**Key ideas**
1. **Jev as a feature extractor.** `Noul` and `Score` turn text into stable numeric columns your existing model can use.
2. **`NoulCriteria` makes a feature stable** by pinning down what yes and no mean ("complaining" is not "leaving").
3. **Evaluate like you mean it:** held-out splits, a baseline, train-only scaling, features that never see the label.
4. **Big gains on toy data are a smell.** Check for labels hiding in the input, timing leaks, and tiny test sets, and compare against a cheap text baseline.

**Exercises**
- Remove the 6 most explicit "cancel"-style messages from the churners and re-run Steps 4–5. How much of Jev's lead survives?
- Add a third Jev feature, e.g. a `Noul` for *"mentions price or budget"*, and check whether it helps the baseline.
- Rewrite five churners' messages to be subtle (no leaving words) and compare `kw_leave` with `jev_leave` on them.
- Replace accuracy with a calibration check: bucket the predicted probabilities and compare each bucket to the actual churn rate.